# Explore the dataset

In [3]:
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS

## Setup

Let us create the graph and create the 3 datasets:

- morphemes
- lexical entries
- word-formation rules

In [15]:
a = RDF.type
ontolex = Namespace('http://www.w3.org/ns/lemon/ontolex#')
der = Namespace('https://github.com/adamfengxx/italian-derivmorph-small/wfr#')

In [2]:
g = Graph()
g.parse('affixes.ttl')
g.parse('./lexical_entries.ttl')
g.parse('./wfr.ttl')

<Graph identifier=N86e360a49ba7410dab5dcf1dae54e9ac (<class 'rdflib.graph.Graph'>)>

Let's bind the namespaces, it might be useful later on for visualization...

In [24]:
ns = {"lex": "https://github.com/adamfengxx/italian-derivmorph-small/lexical_entries#",
"mor": "https://github.com/adamfengxx/italian-derivmorph-small/morph#",
"der": "https://github.com/adamfengxx/italian-derivmorph-small/wfr#",
"morph": "http://www.w3.org/ns/lemon/morph#",
"vartrans": "http://www.w3.org/ns/lemon/vartrans#",
"ontolex": "http://www.w3.org/ns/lemon/ontolex#",
"rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
"rdfs": "http://www.w3.org/2000/01/rdf-schema#"}

for k, v in ns.items():
    g.bind(k, v)

## Inspect the data

How many lexical entries do we have?

In [10]:
lex_entries = [l for l in g.subjects(a, ontolex.LexicalEntry)]
len(lex_entries)

16020

Let's see the triples attached to one of them

In [14]:
for p, o in g.predicate_objects(lex_entries[0]):
    print(p, o)

http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/ns/lemon/ontolex#LexicalEntry
http://www.w3.org/2000/01/rdf-schema#label abbassamento
http://www.w3.org/ns/lemon/ontolex#canonicalForm http://liita.it/data/id/lemma/959890


Here is a word-formation rule

In [16]:
for p, o in g.predicate_objects(der.r_3940):
    print(p, o)

http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/ns/lemon/morph#WordFormationRelation
http://www.w3.org/ns/lemon/vartrans#source https://github.com/adamfengxx/italian-derivmorph-small/lexical_entries#b_460
http://www.w3.org/ns/lemon/vartrans#target https://github.com/adamfengxx/italian-derivmorph-small/lexical_entries#l_1
http://www.w3.org/ns/lemon/morph#wordFormationRule https://github.com/adamfengxx/italian-derivmorph-small/wfr#r_3940_wfp1
http://www.w3.org/ns/lemon/morph#wordFormationRule https://github.com/adamfengxx/italian-derivmorph-small/wfr#r_3940_wfp2
http://www.w3.org/ns/lemon/morph#wordFormationRule https://github.com/adamfengxx/italian-derivmorph-small/wfr#r_3940_wfp3
http://www.w3.org/1999/02/22-rdf-syntax-ns#_1 https://github.com/adamfengxx/italian-derivmorph-small/wfr#r_3940_wfp1
http://www.w3.org/1999/02/22-rdf-syntax-ns#_2 https://github.com/adamfengxx/italian-derivmorph-small/wfr#r_3940_wfp2
http://www.w3.org/1999/02/22-rdf-syntax-ns#_3 https://githu

The whole dataset introduces 1009 forms for:

- morphemes which are *not* in LiITA
- occasional words that don't have a lemma in LiITA

In [53]:
c = 0
for s in g.subjects(a, ontolex.Form):
    c += 1
print(f"Forms that are not in LiITA: {c}")

Forms that are not in LiITA: 1009


Let's try to find out how many of them are Lexical Entries that have not been linked. And since we're at it, let's see how many entries *are* linked. We use SPARQL to query our data:

In [44]:
spql = '''
PREFIX ontolex: <http://www.w3.org/ns/lemon/ontolex#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

select ?s ?f ?lab where {
  ?s a ontolex:LexicalEntry ;
     ontolex:canonicalForm ?f ;
     rdfs:label ?lab
}
'''

In [47]:
res = g.query(spql)

In [50]:
linked = []
not_linked = []

for r in res.bindings:
    if str(r['f']).startswith('http://liita.it/data/id/'):
        linked.append(r)
    else:
        not_linked.append(r)

print('Linked: ', len(linked) )
print('Not linked ', len(not_linked))


Linked:  15128
Not linked  892


So, there are 892 words (i.e. Lexical Entries) that are not linked to a lemma in LiITA.

Let's now inspect the first 10 lemmas that are linked:

In [51]:
for lm in linked[:10]:
    print(lm['lab'])

abbassamento
abbattimento
abbellimento
abbigliamento
abbinamento
abboccamento
abbonamento
abbordabile
abbreviazione
abbrutimento


Let's see now the first 10 lemmas that are *not* linked:

In [52]:
for lm in not_linked[:10]:
    print(lm['lab'])

addentabile
aggiudicatore
anagrafista
anemizzazione
annientatore
annodatore
anti-intercettazione
anti-inflazione
anti-riscaldamento
anti-totalitarista
